# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema and referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadata# Print the dataset metadata: name and descriptionprint(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs, following the Croissant schema. All IDs are given by their `@id` attribute.

In [ ]:
# List the available record sets and their fields, all referenced by @idrecord_sets = [x['@id'] for x in getattr(metadata, 'recordSet', [])]print("Available Record Sets (by @id):")for rs in getattr(metadata, 'recordSet', []):    print(f"  RecordSet @id: {rs['@id']}")    fields = rs.get('field', [])    # Fields can be a dict (single) or a list    if isinstance(fields, dict):        fields = [fields]    print("    Fields:")    for field in fields:        print(f"      Field @id: {field.get('@id')}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Specify the record sets you wish to analyze, by @idrecord_sets_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]dataframes = {}for record_set_id in record_sets_ids:    records_iter = dataset.records(record_set=record_set_id)    # Convert to DataFrame    df = pd.DataFrame(records_iter)    dataframes[record_set_id] = df# As a demonstration, select the first record set's IDif record_sets_ids:    first_rs_id = record_sets_ids[0]    print(f"Columns for RecordSet '{first_rs_id}':", dataframes[first_rs_id].columns.tolist())    display(dataframes[first_rs_id].head())else:    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All references are to field `@id`s.

In [ ]:
# Pick a record set and numeric field @id for analysisif record_sets_ids:    rs_id = first_rs_id    df = dataframes[rs_id].copy()    # Try to detect numeric fields from the schema, fallback to user input    fields = []    for rs in getattr(metadata, 'recordSet', []):        if rs['@id'] == rs_id:            fields = rs.get('field', [])            if isinstance(fields, dict):                fields = [fields]            break    # Find first numeric field (Float, Integer, Number)    numeric_types = {'schema:Float', 'schema:Integer', 'schema:Number', 'Float', 'Integer', 'Number'}    numeric_field_id = None    for field in fields:        dt = field.get('dataType')        if isinstance(dt, dict) and dt.get('@id') in numeric_types:            numeric_field_id = field['@id']            break        elif isinstance(dt, str) and dt in numeric_types:            numeric_field_id = field['@id']            break    if numeric_field_id is not None and numeric_field_id in df.columns:        # Assume the field's values are numeric (not strings)        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')        threshold = df[numeric_field_id].mean()        filtered_df = df[df[numeric_field_id] > threshold]        print(f"Filtered {len(filtered_df)} records in '{rs_id}' with '{numeric_field_id}' > {threshold:.3f}")        display(filtered_df.head())        # Normalize        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-9)        print(f"Normalized '{numeric_field_id}' for filtered records:")        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())        # Try to group by another field (non-numeric, e.g., first non-numeric field)        group_field_id = None        for fld in fields:            dtype = fld.get('dataType')            if fld['@id'] != numeric_field_id and ((isinstance(dtype, str) and dtype not in numeric_types) or (isinstance(dtype, dict) and dtype.get('@id') not in numeric_types)):                if fld['@id'] in filtered_df.columns:                    group_field_id = fld['@id']                    break        if group_field_id:            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")            display(grouped_df.head())        else:            print("No suitable non-numeric group field found.")    else:        print('No numeric field found in this record set.')else:    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references by `@id`.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Simple visualization: histogram of the first numeric fieldif record_sets_ids and 'numeric_field_id' in locals() and numeric_field_id and numeric_field_id in df.columns:    plt.figure(figsize=(8,4))    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)    plt.title(f"Distribution of {numeric_field_id}")    plt.xlabel(numeric_field_id)    plt.ylabel('Count')    plt.show()else:    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize the key findings and observations from the dataset exploration.
- The dataset metadata, record sets, and fields can be inspected by their `@id` fields via the Croissant schema.
- Data records can be loaded and processed into pandas DataFrames for further analytics.
- Numeric fields can be explored, filtered, normalized, and visualized using standard data science libraries.
- Use the list of available fields (with their `@id`) to address domain-specific analyses (e.g., protein quantification, peptide counts, etc.).